# 06 · Graph Q-Network

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

## Imports

In [ ]:
import pickle
import random
import time
import torch
import matplotlib.pyplot as plt

from cdcl import Solver, solve, vsids_literal, random_literal

## Settings

In [ ]:
SEED = 42
DIM = 64
ROUNDS = 4
HEURISTIC_TEST_SIZE = 50

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Q-network file

In [ ]:
%%writefile qnet.py
import random
import torch
import torch.nn as nn


def state_graph(solver):
    free = solver.unassigned()
    index = {v: i for i, v in enumerate(free)}
    edge_var = []
    edge_clause = []
    edge_sign = []
    n_clauses = 0

    for clause in solver.clauses[:solver.n_original]:
        if any(solver.value(lit) is True for lit in clause):
            continue
        free_lits = [lit for lit in clause if solver.value(lit) is None]
        if not free_lits:
            continue
        for lit in free_lits:
            edge_var.append(index[abs(lit)])
            edge_clause.append(n_clauses)
            edge_sign.append(1.0 if lit > 0 else -1.0)
        n_clauses += 1

    return {
        "vars": free,
        "edge_var": torch.tensor(edge_var, dtype=torch.long),
        "edge_clause": torch.tensor(edge_clause, dtype=torch.long),
        "edge_sign": torch.tensor(edge_sign, dtype=torch.float),
        "n_vars": len(free),
        "n_clauses": n_clauses,
    }


def batch_graphs(graphs):
    edge_var = []
    edge_clause = []
    edge_sign = []
    var_graph = []
    var_offset = 0
    clause_offset = 0

    for g, graph in enumerate(graphs):
        edge_var.append(graph["edge_var"] + var_offset)
        edge_clause.append(graph["edge_clause"] + clause_offset)
        edge_sign.append(graph["edge_sign"])
        var_graph += [g] * graph["n_vars"]
        var_offset += graph["n_vars"]
        clause_offset += graph["n_clauses"]

    return {
        "edge_var": torch.cat(edge_var),
        "edge_clause": torch.cat(edge_clause),
        "edge_sign": torch.cat(edge_sign),
        "var_graph": torch.tensor(var_graph, dtype=torch.long),
        "n_vars": var_offset,
        "n_clauses": clause_offset,
        "n_graphs": len(graphs),
    }


def to_device(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def mlp(in_dim, out_dim, hidden):
    return nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.ReLU(),
        nn.Linear(hidden, out_dim),
    )


class QNetwork(nn.Module):
    def __init__(self, dim, rounds):
        super().__init__()
        self.dim = dim
        self.rounds = rounds
        self.var_init = nn.Parameter(torch.randn(1, dim) / dim ** 0.5)
        self.clause_init = nn.Parameter(torch.randn(1, dim) / dim ** 0.5)
        self.var_to_clause = mlp(dim + 1, dim, dim)
        self.clause_update = mlp(2 * dim, dim, dim)
        self.clause_to_var = mlp(dim + 1, dim, dim)
        self.var_update = mlp(2 * dim, dim, dim)
        self.clause_norm = nn.LayerNorm(dim)
        self.var_norm = nn.LayerNorm(dim)
        self.q_head = mlp(dim, 2, dim)

    def forward(self, batch):
        device = self.var_init.device
        edge_var = batch["edge_var"]
        edge_clause = batch["edge_clause"]
        sign = batch["edge_sign"].unsqueeze(1)

        var_h = self.var_init.repeat(batch["n_vars"], 1)
        clause_h = self.clause_init.repeat(batch["n_clauses"], 1)

        for _ in range(self.rounds):
            messages = self.var_to_clause(torch.cat([var_h[edge_var], sign], dim=1))
            clause_in = torch.zeros(batch["n_clauses"], self.dim, device=device).index_add(0, edge_clause, messages)
            clause_h = self.clause_norm(clause_h + self.clause_update(torch.cat([clause_h, clause_in], dim=1)))

            messages = self.clause_to_var(torch.cat([clause_h[edge_clause], sign], dim=1))
            var_in = torch.zeros(batch["n_vars"], self.dim, device=device).index_add(0, edge_var, messages)
            var_h = self.var_norm(var_h + self.var_update(torch.cat([var_h, var_in], dim=1)))

        return self.q_head(var_h)


def literal_from_action(graph, action):
    v = graph["vars"][action // 2]
    return v if action % 2 == 1 else -v


def choose_action(model, graph, epsilon, device):
    if random.random() < epsilon:
        return random.randrange(2 * graph["n_vars"])
    with torch.no_grad():
        q = model(to_device(batch_graphs([graph]), device))
    return q.view(-1).argmax().item()


def q_literal(model, device):
    def choose(solver):
        graph = state_graph(solver)
        return literal_from_action(graph, choose_action(model, graph, 0.0, device))
    return choose

## Import Q-network

In [ ]:
from qnet import QNetwork, state_graph, batch_graphs, to_device, literal_from_action, choose_action, q_literal

## Load data

In [ ]:
with open("rl_data.pkl", "rb") as f:
    rl_data = pickle.load(f)

rl_train = rl_data["train"]
rl_val = rl_data["val"]

print("Training formulas:  ", len(rl_train))
print("Validation formulas:", len(rl_val))

## State graph of a new formula

In [ ]:
problem = rl_train[0]
solver = Solver(problem["n_vars"], problem["clauses"])
solver.start()

graph = state_graph(solver)

print("Formula variables:  ", problem["n_vars"])
print("Formula clauses:    ", len(problem["clauses"]))
print("Graph variables:    ", graph["n_vars"])
print("Graph clauses:      ", graph["n_clauses"])
print("Graph edges:        ", len(graph["edge_var"]))
print("First edges (var, clause, sign):")
for i in range(5):
    print("  ", graph["edge_var"][i].item(), graph["edge_clause"][i].item(), graph["edge_sign"][i].item())

## State graph after a few decisions

In [ ]:
for _ in range(3):
    if solver.result is None:
        solver.step(vsids_literal(solver))

if solver.result is None:
    graph = state_graph(solver)
    print("Decisions made:  ", solver.decisions)
    print("Assigned vars:   ", len(solver.assignment))
    print("Graph variables: ", graph["n_vars"])
    print("Graph clauses:   ", graph["n_clauses"])
    print("Graph edges:     ", len(graph["edge_var"]))
else:
    print("Formula already solved:", solver.result)

## Graph size during one episode

In [ ]:
problem = max(rl_val, key=lambda p: p["n_vars"])
solver = Solver(problem["n_vars"], problem["clauses"])
solver.start()

var_counts = []
clause_counts = []
while solver.result is None:
    graph = state_graph(solver)
    var_counts.append(graph["n_vars"])
    clause_counts.append(graph["n_clauses"])
    solver.step(vsids_literal(solver))

plt.figure(figsize=(7, 4))
plt.plot(var_counts, label="Unassigned variables")
plt.plot(clause_counts, label="Open clauses")
plt.xlabel("Decision")
plt.ylabel("Nodes in graph")
plt.title(f"n = {problem['n_vars']}, result = {solver.result}")
plt.legend()
plt.tight_layout()
plt.savefig("graph_size_episode.png", dpi=150)
plt.show()

## Batching several states

In [ ]:
graphs = []
for problem in rl_train[:4]:
    solver = Solver(problem["n_vars"], problem["clauses"])
    solver.start()
    if solver.result is None:
        graphs.append(state_graph(solver))

batch = batch_graphs(graphs)

print("Graphs in batch: ", batch["n_graphs"])
print("Variables:       ", batch["n_vars"], [g["n_vars"] for g in graphs])
print("Clauses:         ", batch["n_clauses"], [g["n_clauses"] for g in graphs])
print("Edges:           ", len(batch["edge_var"]))
print("Variable to graph:", batch["var_graph"].tolist()[:20], "...")

## Build the network

In [ ]:
model = QNetwork(DIM, ROUNDS).to(device)

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Forward pass

In [ ]:
q = model(to_device(batch, device))

print("Q-values shape:", tuple(q.shape))
print("Expected:      ", (batch["n_vars"], 2))
print("First rows:")
print(q[:5])

## Q-values for one state

In [ ]:
graph = graphs[0]
with torch.no_grad():
    q = model(to_device(batch_graphs([graph]), device)).view(-1)

top = torch.topk(q, 5)
print("Top 5 actions of the untrained network:")
for value, action in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  action {action:3d} | literal {literal_from_action(graph, action):4d} | Q {value:.4f}")

## Epsilon-greedy actions

In [ ]:
greedy_action = choose_action(model, graph, 0.0, device)

for epsilon in [0.0, 0.1, 0.5, 1.0]:
    actions = [choose_action(model, graph, epsilon, device) for _ in range(500)]
    share_greedy = sum(a == greedy_action for a in actions) / len(actions)
    print(f"epsilon {epsilon:.1f} | share of greedy action {share_greedy:.2f} | distinct actions {len(set(actions))}")

## Untrained network as a branching heuristic

In [ ]:
model.eval()
test_formulas = rl_val[:HEURISTIC_TEST_SIZE]

heuristics = {
    "VSIDS": vsids_literal,
    "Random": random_literal,
    "Untrained Q": q_literal(model, device),
}

heuristic_results = {}
for name, choose in heuristics.items():
    random.seed(SEED)
    decisions = []
    correct = 0
    for p in test_formulas:
        solver = solve(p["n_vars"], p["clauses"], choose)
        decisions.append(solver.decisions)
        correct += solver.result == bool(p["label"])
    heuristic_results[name] = decisions
    print(f"{name:12s} | mean decisions {sum(decisions) / len(decisions):6.1f} | correct {correct}/{len(test_formulas)}")

## Decisions per formula

In [ ]:
plt.figure(figsize=(7, 4))
plt.boxplot(list(heuristic_results.values()), labels=list(heuristic_results.keys()))
plt.ylabel("Decisions per formula")
plt.tight_layout()
plt.savefig("untrained_q_decisions.png", dpi=150)
plt.show()

## Decision time

In [ ]:
problem = rl_val[0]
choose_q = q_literal(model, device)

for name, choose in [("VSIDS", vsids_literal), ("Q-network", choose_q)]:
    solver = Solver(problem["n_vars"], problem["clauses"])
    solver.start()
    if solver.result is not None:
        print("Formula solved without decisions")
        break
    start = time.perf_counter()
    for _ in range(100):
        choose(solver)
    elapsed = (time.perf_counter() - start) / 100
    print(f"{name:10s} | {elapsed * 1000:.3f} ms per decision")

## Save

In [ ]:
torch.save(model.state_dict(), "qnet_untrained.pt")

print("Saved qnet.py, qnet_untrained.pt and the plots")